# Dense HEA LeJEPA on Synthetic Disc-Square Data (DDP)

Same recipe as `hea_dense_lejepa_drive_features.ipynb`, with **optional multi-GPU**
training via `torch.distributed` + `DistributedDataParallel`.

- **Single process** (default in Jupyter): same behavior as the original notebook.
- **Multi-GPU (CUDA)**: the training cells are written so that when `torchrun`
  sets `WORLD_SIZE`, `RANK`, and `LOCAL_RANK`, NCCL is initialized and the model
  is wrapped in `DDP`. Typical launch for a **Python script** that contains the
  same logic:

```bash
# From repo root; set nproc_per_node to GPUs on the machine
torchrun --standalone --nproc_per_node=4 path/to/train_dense_lejepa_ddp.py
```

Interactive Jupyter runs **one Python process**. The first code cell detects a
Jupyter/IPython kernel and **ignores `WORLD_SIZE>1` in the environment** (common on
HPC login nodes) so **Run All** works like the non-DDP notebook. Export to a `.py`
and use `torchrun` for true multi-GPU. To attempt DDP from a notebook anyway, set
`ALLOW_JUPYTER_DDP=1` (you must use a launcher that starts one process per GPU).

For **multi-GPU from a notebook** (spawn), see the section **Multi-GPU training
from this notebook** and `examples/dense_lejepa_ddp_spawn.py`.


In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from local_conv_attention import (
    DenseLeJEPAModel,
    DiscSquareDataset,
    default_all_latent_hooks,
    generate_disc_square_image,
    load_experiment_config,
    make_disc_square_types,
)


def _launched_from_ipython_kernel() -> bool:
    """True inside Jupyter / IPython (typical single-process kernel)."""
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is None:
            return False
        return ip.__class__.__name__ in ("ZMQInteractiveShell", "TerminalInteractiveShell")
    except Exception:
        return False


def setup_distributed() -> tuple[bool, int, int, int]:
    """DDP when launched with torchrun / Slurm. Safe to re-run in Jupyter (idempotent)."""
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))

    allow_jupyter_ddp = os.environ.get("ALLOW_JUPYTER_DDP", "").strip().lower() in (
        "1",
        "true",
        "yes",
    )
    if world_size > 1 and _launched_from_ipython_kernel() and not allow_jupyter_ddp:
        if rank == 0:
            print(
                "[setup] Jupyter kernel + WORLD_SIZE>1 in env -> using single-process mode.\n"
                "        Set ALLOW_JUPYTER_DDP=1 only if you use a multi-rank notebook launcher.\n"
                "        For multi-GPU training, prefer a .py script with torchrun."
            )
        world_size, rank, local_rank = 1, 0, 0

    distributed = world_size > 1

    if dist.is_initialized():
        same_group = (
            distributed
            and dist.get_world_size() == world_size
            and dist.get_rank() == rank
        )
        if not same_group:
            dist.destroy_process_group()

    if distributed:
        if not torch.cuda.is_available():
            raise RuntimeError("Multi-GPU DDP requires CUDA (set CUDA_VISIBLE_DEVICES).")
        torch.cuda.set_device(local_rank)
        if not dist.is_initialized():
            dist.init_process_group(backend="nccl", init_method="env://")

    return distributed, rank, world_size, local_rank


DISTRIBUTED, RANK, WORLD_SIZE, LOCAL_RANK = setup_distributed()
IS_MAIN = RANK == 0

if DISTRIBUTED:
    device = torch.device(f"cuda:{LOCAL_RANK}")
else:
    device = torch.device(
        "mps"
        if torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
    )

PIN_MEMORY = device.type == "cuda"

if IS_MAIN:
    print("Project root:", PROJECT_ROOT)
    print("Device:", device)
    if DISTRIBUTED:
        print(f"DDP active | world_size={WORLD_SIZE} rank={RANK} local_rank={LOCAL_RANK}")

# Same initial weights on every rank; different batches come from DistributedSampler.
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)


## Multi-GPU training from this notebook (`torch.multiprocessing.spawn`)

If another project ran DDP **from a notebook**, it almost certainly spawned **separate
Python processes** (one per GPU)—not the single Jupyter kernel. The Jupyter kernel
still shows `Device: cuda` for one GPU; training runs in child processes.

This repo ships [`examples/dense_lejepa_ddp_spawn.py`](dense_lejepa_ddp_spawn.py), which
calls `torch.multiprocessing.spawn`. Workers are *not* IPython, so they do **not** hit
the “ignore WORLD_SIZE in Jupyter” guard in the first cell.

**Requirements:** 2+ visible GPUs on the node (`CUDA_VISIBLE_DEVICES`), repo root on
`sys.path` (the first cell sets `PROJECT_ROOT`).

Skip the later **single-process** training cell if you use the cell below (or you will
train twice).


In [ ]:
# Uncomment to run multi-GPU training from the notebook (spawn).
# Needs 2+ GPUs. Skip the single-process `train_dense_lejepa_ddp(...)` cell below if you run this.

# from examples.dense_lejepa_ddp_spawn import run_spawn_training

# history_mp = run_spawn_training(PROJECT_ROOT, epochs=50, batch_size=8)
# history_mp


In [ ]:
canonical_specs = make_disc_square_types()
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, spec in zip(axes.flat, canonical_specs):
    image = generate_disc_square_image(spec)[0]
    axis.imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
    axis.set_title(
        f"r={spec.radius}, s={spec.square_size}, near={spec.near_density}, far={spec.far_density}",
        fontsize=9,
    )
    axis.axis("off")
plt.tight_layout()

In [ ]:
train_dataset = DiscSquareDataset(repeats_per_type=256)

if DISTRIBUTED:
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=WORLD_SIZE,
        rank=RANK,
        shuffle=True,
        drop_last=True,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        sampler=train_sampler,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )
else:
    train_sampler = None
    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )

if IS_MAIN:
    print("Dataset size:", len(train_dataset))
    sample = train_dataset[0]
    print("Sample keys:", sample.keys())
    print("Sample image shape:", tuple(sample["image"].shape))
    print(
        "Sample metadata:",
        {
            "type_index": sample["type_index"],
            "radius": sample["radius"],
            "square_size": sample["square_size"],
            "near_density": sample["near_density"],
            "far_density": sample["far_density"],
        },
    )


In [ ]:
config = load_experiment_config(PROJECT_ROOT / "configs" / "hea_dense_lejepa_default.yaml")
config.model.in_channels = 1
config.model.base_channels = 16
config.model.channel_multipliers = [1, 2, 4, 8]
config.model.encoder_depths = [1, 1, 1, 1]
config.model.decoder_depths = [1, 1, 1]
config.model.attention.heads = 4
config.model.attention.head_dim = 16
config.model.attention.operator_backend = "optimized"

# For this synthetic task we want local detail, so we read latents from the
# highest-resolution encoder stage rather than the semantically smoother top decoder map.
config.model.latent.sources = default_all_latent_hooks(len(config.model.channel_multipliers))
config.model.latent.step_mode = "rotate"
config.model.latent.latent_dim = 32
config.model.latent.projector_depth = 1
config.model.latent.normalize_latents = False

# Keep HEA present but lightweight.
config.model.hea.enabled_decoder_stages = [0]
config.model.semantic_memory.window_sizes = [7, 7, 7]
config.model.hea.per_scale_window_sizes = [7, 7, 7]

# Symmetric dense LeJEPA settings.
config.model.lejepa.num_views = 4
# Lower VRAM: one view through backbone at a time (see lejepa.sequential_view_forward).
config.model.lejepa.sequential_view_forward = True
config.model.lejepa.lambda_sigreg = 0.05
config.model.lejepa.sigreg.enabled = True
config.model.lejepa.sigreg.num_slices = 64
config.model.lejepa.sigreg.num_knots = 17
config.model.lejepa.sigreg.per_view = True

# Geometry stays aligned; augmentations are photometric/corruption only.
config.model.lejepa.views.mode = "aligned_same_geometry"
config.model.lejepa.views.corruption.intensity_jitter = True
config.model.lejepa.views.corruption.blur = False
config.model.lejepa.views.corruption.gaussian_noise = True
config.model.lejepa.views.corruption.random_block_mask = True
config.model.lejepa.views.corruption.block_mask_ratio = 0.04
config.model.lejepa.views.corruption.block_mask_num_blocks = 20

config.validate()

model = DenseLeJEPAModel(config.model).to(device)
if DISTRIBUTED:
    # If latent.sources is a strict subset (e.g. only encoder_0), set find_unused True.
    model = DDP(
        model,
        device_ids=[LOCAL_RANK],
        output_device=LOCAL_RANK,
        find_unused_parameters=True,
    )
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def unwrap_model(m):
    return m.module if isinstance(m, DDP) else m


if IS_MAIN:
    num_params = sum(p.numel() for p in unwrap_model(model).parameters())
    print(f"Model parameters: {num_params:,}")


In [ ]:
def train_dense_lejepa_ddp(
    model,
    loader,
    optimizer,
    *,
    epochs: int = 20,
    sampler: DistributedSampler | None = None,
) -> dict[str, list[float]]:
    history: dict[str, list[float]] = {"loss": [], "inv_loss": [], "sigreg_loss": []}
    global_step = 0
    for epoch in range(epochs):
        if sampler is not None:
            sampler.set_epoch(epoch)
        model.train()
        running = {key: 0.0 for key in history}
        batches = 0
        for batch in loader:
            images = batch["image"].to(device, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            out = model(images, rotate_latent_index=global_step)
            global_step += 1
            out["loss"].backward()
            optimizer.step()
            for key in history:
                running[key] += float(out[key].detach())
            batches += 1

        if DISTRIBUTED:
            stats = torch.tensor(
                [running["loss"], running["inv_loss"], running["sigreg_loss"], float(batches)],
                device=device,
                dtype=torch.float64,
            )
            dist.all_reduce(stats, op=dist.ReduceOp.SUM)
            total_batches = max(int(stats[3].item()), 1)
            for i, key in enumerate(list(history.keys())):
                history[key].append(float(stats[i].item() / total_batches))
        else:
            for key in history:
                history[key].append(running[key] / max(1, batches))

        if IS_MAIN:
            print(
                f"epoch {epoch + 1:02d} | loss={history['loss'][-1]:.4f} | "
                f"inv={history['inv_loss'][-1]:.4f} | sigreg={history['sigreg_loss'][-1]:.4f}"
            )
    return history


history = train_dense_lejepa_ddp(
    model, train_loader, optimizer, epochs=50, sampler=train_sampler
)


In [ ]:
if IS_MAIN:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].plot(history["loss"], marker="o")
    axes[0].set_title("Total loss")
    axes[1].plot(history["inv_loss"], marker="o")
    axes[1].set_title("Dense invariance")
    axes[2].plot(history["sigreg_loss"], marker="o")
    axes[2].set_title("SIGReg")
    for ax in axes:
        ax.set_xlabel("epoch")
    plt.tight_layout()


In [ ]:
@torch.no_grad()
def extract_dense_latent_map(core_model, image: torch.Tensor) -> torch.Tensor:
    core_model.eval()
    image = image.unsqueeze(0).to(device)
    features = core_model.backbone(image)
    latents = core_model.projector(features)
    return latents[0].cpu()


def pca_colorize_latents(latent_map: torch.Tensor) -> np.ndarray:
    channels, height, width = latent_map.shape
    flat = latent_map.reshape(channels, -1).T.numpy()
    flat = flat - flat.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(flat, full_matrices=False)
    coords = flat @ vh[:3].T
    coords = (coords - coords.min(axis=0, keepdims=True)) / (
        coords.max(axis=0, keepdims=True) - coords.min(axis=0, keepdims=True) + 1e-6
    )
    return coords.reshape(height, width, 3)


if IS_MAIN:
    eval_model = unwrap_model(model)
    eval_sample = train_dataset[0]
    image = eval_sample["image"]
    latent_map = extract_dense_latent_map(eval_model, image)
    latent_norm = torch.linalg.vector_norm(latent_map, dim=0)
    latent_pca = pca_colorize_latents(latent_map)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image[0], cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title("Synthetic image")
    axes[1].imshow(latent_norm, cmap="magma")
    axes[1].set_title("Latent norm")
    axes[2].imshow(latent_pca)
    axes[2].set_title("Latent PCA color")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    print("Dense latent map shape:", tuple(latent_map.shape))


In [ ]:
@torch.no_grad()
def build_canonical_feature_bank(core_model) -> tuple[np.ndarray, list[dict]]:
    pooled_features = []
    metadata = []
    for spec in canonical_specs:
        image = generate_disc_square_image(spec)
        latent = extract_dense_latent_map(core_model, image)
        pooled = latent.mean(dim=(1, 2))
        pooled_features.append(pooled.numpy())
        metadata.append(
            {
                "type_index": spec.type_index,
                "radius": spec.radius,
                "square_size": spec.square_size,
                "near_density": spec.near_density,
                "far_density": spec.far_density,
            }
        )
    return np.stack(pooled_features), metadata


if IS_MAIN:
    eval_model = unwrap_model(model)
    feature_bank, feature_meta = build_canonical_feature_bank(eval_model)
    feature_bank_centered = feature_bank - feature_bank.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(feature_bank_centered, full_matrices=False)
    feature_pca = feature_bank_centered @ vh[:2].T

    plt.figure(figsize=(6, 5))
    for point, meta in zip(feature_pca, feature_meta):
        plt.scatter(point[0], point[1], s=80)
        plt.text(point[0] + 0.01, point[1] + 0.01, str(meta["type_index"]), fontsize=9)
    plt.title("Pooled latent PCA for canonical types")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()

    similarity = feature_bank @ feature_bank.T
    plt.figure(figsize=(5, 4))
    plt.imshow(similarity, cmap="viridis")
    plt.title("Canonical feature similarity")
    plt.colorbar()
    plt.tight_layout()


In [ ]:
if IS_MAIN:
    try:
        import napari

        v = napari.Viewer()
        v.add_image(latent_map)
    except NameError:
        print("Skipping napari: run the visualization cell above first.")
    except ImportError:
        print("napari not installed; skipping interactive viewer.")


In [ ]:
# Optional: tear down process group after training (useful in batch scripts).
# if DISTRIBUTED and dist.is_initialized():
#     dist.barrier()
#     dist.destroy_process_group()
